# Notebook A — Bank Customer Churn and Bank Marketing

**Hybrid quantum–classical classification with Grey Wolf Optimizer hyper-parameter search**
Datasets: **Bank Customer Churn, Bank Marketing**

This notebook produces the results for these two datasets reported in the manuscript
*Optimizing Quantum Machine Learning for Business Analytics: A Hybrid Grey Wolf Optimizer Framework*. It runs:

1. repeated stratified nested cross-validation (5 folds × 3 repeats, inner 3-fold search), with all preprocessing fitted inside each training fold;
2. Grey Wolf Optimizer search for four quantum and six classical classifiers, plus default-hyper-parameter runs on the same folds;
3. paired statistical tests (Wilcoxon, Holm, Cliff's δ, Friedman–Nemenyi);
4. GWO versus Particle Swarm Optimisation and Random Search at an equal evaluation budget;
5. a gradient-variance scan and an Adam / SPSA / COBYLA comparison;
6. a class-imbalance ablation, SHAP attributions and the figures.

**Run order:** execute cells top to bottom. Section 3 is the expensive one; set
`CONFIG = SMOKE` first to verify the whole notebook runs in a few minutes, then
switch to `FULL` for the reported results.

> **Resumable.** Every expensive section writes its result to disk and
> skips itself on a later run if that result already exists. After the full run
> has completed once, **Run All** only recomputes the fast steps (tables,
> statistics, figures) plus anything whose output file is missing — it takes
> minutes, not hours. To force a section to recompute, delete its output file
> (named in that cell's first comment).

## 1. Environment and configuration

In [ ]:
# Run once per environment.
# !pip install -q "pennylane>=0.40" scikit-learn pandas numpy scipy matplotlib \
#                 imbalanced-learn xgboost shap scikit-posthocs joblib

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from qmlgwo.config import FULL, SMOKE, ExperimentConfig, seed_everything, environment_manifest
from qmlgwo.data import default_specs, dataset_summary, build_pipeline
from qmlgwo.evaluate import (build_registry, run_matrix, nested_cv, summarise,
                             format_table, aggregate_confusion, load_folds)
from qmlgwo.stats import paired_comparison, friedman_nemenyi, add_bootstrap_cis
from qmlgwo.diagnostics import (gradient_variance_scan, fit_decay,
                                optimizer_trajectories, plateau_avoidance_evidence,
                                summarise_trajectories)
from qmlgwo import viz

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 60)

# ---------------------------------------------------------------- #
# SMOKE = fast structural check (minutes).  FULL = reported results. #
# ---------------------------------------------------------------- #
CONFIG = FULL           # reported results. Use SMOKE for a fast structural check.

NOTEBOOK   = "A"
# dataset paths are declared in the next section (DATA_ROOT / SPECS)
RESULTS    = Path(f"results/notebook_{NOTEBOOK}"); RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES    = Path(f"figures/notebook_{NOTEBOOK}"); FIGURES.mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG.master_seed)
CONFIG.save(RESULTS / "config_and_environment.json")

print(f"master seed        : {CONFIG.master_seed}")
print(f"outer protocol     : {CONFIG.n_outer_splits}-fold x {CONFIG.n_outer_repeats} repeats "
      f"= {CONFIG.n_outer_folds} estimates per configuration")
print(f"inner protocol     : {CONFIG.n_inner_splits}-fold (hyper-parameter selection only)")
print(f"HPO budget         : {CONFIG.hpo_budget} objective evaluations per searcher")
print(f"PennyLane / sklearn: {environment_manifest()['packages']['pennylane']} / "
      f"{environment_manifest()['packages']['sklearn']}")

## 2. Datasets

All datasets are **publicly available benchmarks**. Place the files under `data/` as
described in `data/README.md`, or point `DATA_ROOT` below at your own folder.

`duration` is dropped from Bank Marketing because it is recorded only *after* the call
concludes and therefore leaks the target, as the manuscript explains; the
spec makes it machine-checkable via the `leakage_columns` field.

In [ ]:
KEYS = ["bank_churn", "bank_marketing"]

from pathlib import Path
from qmlgwo.data import DatasetSpec, dataset_summary

# ---------------------------------------------------------------------- #
# Local dataset root -- edit this one line if the drive/folder changes.   #
# ---------------------------------------------------------------------- #
DATA_ROOT = Path("data")

SPECS = {
    "bank_churn": DatasetSpec(
        key="bank_churn", display_name="Bank Customer Churn",
        path=str(DATA_ROOT / "Bank Customer Churn Prediction" /
                 "Bank Customer Churn Prediction.csv"),
        target="churn", positive_label=1,
        drop_columns=("customer_id",),
        source="Kaggle: Bank Customer Churn Prediction (public)",
    ),
    "bank_marketing": DatasetSpec(
        key="bank_marketing", display_name="Bank Marketing",
        # bank.csv (raw), NOT bank_processed.csv: the latter is label-encoded,
        # which imposes a false ordinal scale on job / month / poutcome. The
        # pipeline performs one-hot encoding and rare-level grouping per fold,
        # as the manuscript describes.
        path=str(DATA_ROOT / "bank-marketing-uci" / "bank.csv"),
        target="y", positive_label="yes",
        leakage_columns=("duration",),   # known only after the call ends
        source="UCI ML Repository: Bank Marketing (bank.csv, 10% subset, public)",
    ),
    "hr_promotion": DatasetSpec(
        key="hr_promotion", display_name="HR Promotion",
        path=str(DATA_ROOT / "HR Analytics Employee Promotion Data" / "train.csv"),
        target="is_promoted", positive_label=1,
        drop_columns=("employee_id",),
        source="Kaggle/AV: HR Analytics Employee Promotion (public)",
    ),
    "loan_approval": DatasetSpec(
        key="loan_approval", display_name="Loan Approval",
        path=str(DATA_ROOT / "Loan Prediction Problem Dataset" /
                 "train_u6lujuX_CVtuZ9i.csv"),
        target="Loan_Status", positive_label="Y",
        drop_columns=("Loan_ID",),
        source="Analytics Vidhya: Loan Prediction Problem (public)",
    ),
}


def load_bank_marketing_raw(path):
    """Robust reader for bank.csv.

    Some redistributions wrap each line in double quotes, so pandas returns a
    single column. We detect that and split manually. The target is mapped
    explicitly (== "yes") rather than by order of appearance in the file --
    an order-based mapping silently inverts the classes if the first row is a
    positive case, which would compute F1 on the wrong class.
    """
    df = pd.read_csv(path, sep=";", quotechar='"', skipinitialspace=True)
    if df.shape[1] == 1:
        col = df.columns[0]
        cols = [c.strip().strip('"') for c in col.split(";")]
        rows_ = [[v.strip().strip('"') for v in str(r).split(";")] for r in df[col]]
        df = pd.DataFrame(rows_, columns=cols)
    df.columns = [c.strip().strip('"') for c in df.columns]

    for c in ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    y = (df["y"].astype(str).str.strip().str.lower() == "yes").astype(int)
    X = df.drop(columns=["y", "duration"])
    return X.reset_index(drop=True), y.reset_index(drop=True).rename("target")


# Files are used in full: class balancing happens
# INSIDE the CV pipeline (RandomUnderSampler), never before the split, so no
# information crosses between folds.
DATA, rows = {}, []
for k in KEYS:
    spec = SPECS[k]
    if k == "bank_marketing":
        X, y = load_bank_marketing_raw(Path(spec.path))
    else:
        X, y = spec.load()

    DATA[k] = (X, y.to_numpy())
    r = dataset_summary(X, y, spec)
    pos = int(y.sum()); neg = len(y) - pos
    r["After undersampling"] = 2 * min(pos, neg)   # balanced size of the full file
    rows.append(r)

dataset_table = pd.DataFrame(rows)
dataset_table.to_csv(RESULTS / "table_datasets.csv", index=False)
dataset_table

## 3. Experimental protocol

### 3.1 What the GWO actually searches

The two tables below give the complete search space with lower and upper bounds, the
population size, the stopping criterion and the initialisation strategy (manuscript Tables 3
and 4). They are printed directly from the code that runs, so the manuscript and the
implementation cannot diverge.

In [ ]:
from qmlgwo.optim import GreyWolfOptimizer
from qmlgwo.config import get_rng

REGISTRY = build_registry(CONFIG)

# --- GWO settings (manuscript table) ---
gwo_settings = GreyWolfOptimizer(REGISTRY["VQC"].space, CONFIG.gwo_n_wolves,
                                 CONFIG.gwo_n_iterations, CONFIG.gwo_a_initial,
                                 CONFIG.gwo_patience, CONFIG.gwo_tol,
                                 rng=get_rng("gwo")).describe()
gwo_settings.to_csv(RESULTS / "table_gwo_settings.csv", index=False)
print(gwo_settings.to_string(index=False))

# --- per-model search spaces (manuscript table) ---
spaces = []
for name, spec in REGISTRY.items():
    if spec.space is None:
        continue
    df = spec.space.to_frame()
    df.insert(0, "Model", name)
    spaces.append(df)
search_space_table = pd.concat(spaces, ignore_index=True)
search_space_table.to_csv(RESULTS / "table_search_space.csv", index=False)
search_space_table

### 3.2 Nested cross-validation

```
for each outer fold (repeated stratified CV):          <- reporting only
    for each inner fold on the outer TRAIN split:      <- model selection only
        fit -> score
    select hyper-parameters maximising mean inner F1
    refit on full outer TRAIN, score once on outer TEST
```

The outer test fold is used **exactly once**, for reporting.

Running every model with `optimizer=None` **and** `optimizer='gwo'` on the *same* folds is
what makes the paired Wilcoxon test in Section 4 valid.

In [ ]:
MODELS = ["LogReg", "SVC", "DecisionTree", "RandomForest", "MLP", "XGBoost",
          "VQC", "QNN", "QSVC", "QDT"]
MODELS = [m for m in MODELS if m in REGISTRY]

# LDA on a binary target yields exactly C - 1 = 1 component; PCA keeps CONFIG.pca_components.
REDUCTIONS = (("pca", CONFIG.pca_components), ("lda", CONFIG.lda_components))

# ---------------------------------------------------------------------- #
# Checkpointed execution.                                                 #
# Each (dataset, model, reduction, optimizer) combination is written to    #
# disk the moment it finishes. Re-running this cell after an interruption  #
# skips whatever already completed and resumes from there, so a crash at   #
# hour 7 costs one combination rather than the whole run.                  #
# To force a clean re-run, delete the `checkpoints` folder.                #
# ---------------------------------------------------------------------- #
CKPT = RESULTS / "checkpoints"; CKPT.mkdir(parents=True, exist_ok=True)

fold_results = {}
for k in KEYS:
    X, y = DATA[k]
    name = SPECS[k].display_name
    print(f"\n=== {name} ===", flush=True)
    parts, t_ds = [], time.perf_counter()

    # If this dataset already has a complete folds file (e.g. from an earlier run
    # made before checkpointing existed, or in another folder), restore missing
    # checkpoints from it instead of retraining. Re-training is NOT bit-for-bit
    # identical (differences up to ~0.005 F1 were observed), so reusing the
    # original results keeps the tables consistent with the manuscript.
    seed_src = next((p for p in (RESULTS / f"folds_{k}.csv", RESULTS / "folds_all.csv")
                     if p.exists()), None)
    seed_df = load_folds(seed_src) if seed_src is not None else None
    if seed_df is not None and "dataset" in seed_df.columns:
        seed_df = seed_df[seed_df.dataset == name]
    if seed_df is not None:
        print(f"  restore source: {seed_src.name} ({len(seed_df)} rows for {name})", flush=True)

    for reduction, n_comp in REDUCTIONS:
        for opt in (None, "gwo"):
            for m in MODELS:
                spec = REGISTRY[m]
                if opt is not None and spec.space is None:
                    continue                       # nothing to tune
                tag = f"{k}__{m}__{reduction}__{opt or 'default'}"
                fpath = CKPT / f"{tag}.csv"

                if fpath.exists():
                    parts.append(load_folds(fpath))   # restores dict/list columns
                    print(f"  [skip] {tag}", flush=True)
                    continue

                if seed_df is not None:
                    prev = seed_df[(seed_df.model == m)
                                   & (seed_df.reduction == reduction.upper())
                                   & (seed_df.optimizer == (opt or "default"))]
                    if len(prev) == CONFIG.n_outer_folds and (prev.status == "ok").all():
                        prev.to_csv(fpath, index=False)
                        parts.append(prev)
                        print(f"  [restored] {tag}", flush=True)
                        continue

                t0 = time.perf_counter()
                d = nested_cv(X, y, spec, CONFIG, reduction=reduction,
                              n_components=n_comp, optimizer=opt,
                              n_jobs=CONFIG.n_jobs, verbose=False)
                d["dataset"] = name
                d.to_csv(fpath, index=False)       # persist immediately
                parts.append(d)
                print(f"  [ok]   {tag:<48s} "
                      f"F1={d.f1.mean():.3f} AUC={d.roc_auc.mean():.3f} "
                      f"({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)

    df = pd.concat(parts, ignore_index=True)
    fold_results[k] = df
    df.to_csv(RESULTS / f"folds_{k}.csv", index=False)
    print(f"  {name}: {len(df)} fold-records | failures: {(df.status != 'ok').sum()} "
          f"| {(time.perf_counter() - t_ds) / 60:.1f} min", flush=True)

ALL_FOLDS = pd.concat(fold_results.values(), ignore_index=True)
ALL_FOLDS.to_csv(RESULTS / "folds_all.csv", index=False)
print(f"\nTotal fold-level records: {len(ALL_FOLDS)}")
print(f"Failures: {(ALL_FOLDS.status != 'ok').sum()}  (must be 0)")

### 3.3 Results tables

Every cell is `mean ± std` over the outer folds, with a BCa bootstrap 95% CI — never a
single number. If two configurations produce *identical* values here it now means they are
genuinely identical, not that the wrong table was pasted in.

In [ ]:
summary_tables, formatted_tables = {}, {}
for k in KEYS:
    name = SPECS[k].display_name
    s = summarise(fold_results[k])
    # summarise() already carries a normal-approximation CI; the bootstrap one is
    # renamed so both survive the merge and can be compared.
    ci = (add_bootstrap_cis(fold_results[k], "f1", n_boot=CONFIG.n_bootstrap,
                            alpha=CONFIG.alpha, rng=get_rng("cv_outer"))
          .rename(columns={"f1_ci95": "f1_ci95_bca"}))
    s = s.merge(ci[["model", "reduction", "optimizer", "f1_ci95_bca"]],
                on=["model", "reduction", "optimizer"], how="left")
    summary_tables[k] = s
    s.to_csv(RESULTS / f"summary_{k}.csv", index=False)

    f = format_table(s)
    f["F1 95% CI (BCa)"] = s["f1_ci95_bca"].values
    formatted_tables[k] = f
    f.to_csv(RESULTS / f"table_results_{k}.csv", index=False)
    print(f"\n=== {name} ===")
    print(f.to_string(index=False))

In [ ]:
# Effect of GWO, per model: paired difference on identical folds.
gwo_effect = []
for k in KEYS:
    df = fold_results[k]
    wide = df.pivot_table(index=["fold", "model", "reduction"],
                          columns="optimizer", values="f1").dropna()
    if {"default", "gwo"}.issubset(wide.columns):
        d = (wide["gwo"] - wide["default"]).groupby(level=["model", "reduction"]).agg(
            ["mean", "std", "count"])
        d.columns = ["delta_f1_mean", "delta_f1_std", "n_folds"]
        d["dataset"] = SPECS[k].display_name
        gwo_effect.append(d.reset_index())

gwo_effect = pd.concat(gwo_effect, ignore_index=True) if gwo_effect else pd.DataFrame()
if not gwo_effect.empty:
    gwo_effect.to_csv(RESULTS / "table_gwo_effect.csv", index=False)
    print(gwo_effect.sort_values("delta_f1_mean", ascending=False).to_string(index=False))

## 4. Statistical validation

Tests used in the manuscript (Section 3.10):

- **Wilcoxon signed-rank**, paired across outer folds (Demsar, 2006)
- **Holm–Bonferroni** step-down correction for the multiple comparisons
- **Cliff's δ** effect size — significance without magnitude is not a finding
- **Friedman + Nemenyi** omnibus ranking with a critical-difference diagram

In [ ]:
stat_tests = {}
for k in KEYS:
    df = fold_results[k]
    best = (summary_tables[k].iloc[0]["model"], summary_tables[k].iloc[0]["reduction"],
            summary_tables[k].iloc[0]["optimizer"])
    baseline = " | ".join(map(str, best))
    pc = paired_comparison(df, metric="f1", alpha=CONFIG.alpha, baseline=baseline)
    stat_tests[k] = pc
    pc.to_csv(RESULTS / f"stats_pairwise_{k}.csv", index=False)
    print(f"\n=== {SPECS[k].display_name} — vs best config ({baseline}) ===")
    if not pc.empty:
        print(pc[["config_B", "mean_diff", "p_raw", "p_holm",
                  "cliffs_delta", "effect_size", "significant"]].head(12).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------- #
# Friedman omnibus test + Nemenyi critical-difference diagram (Figure 8).  #
#                                                                          #
# The omnibus test is also run over ALL configurations and saved, but the  #
# DIAGRAM is drawn for a shortlist: the four quantum models plus the three #
# strongest classical baselines, LDA projection, GWO-optimised.            #
# With ~32 configurations over 15 folds the critical difference spans      #
# most of the rank axis and the diagram carries no information.            #
# ---------------------------------------------------------------------- #
from scipy import stats

QUANTUM = ["QDT", "QNN", "QSVC", "VQC"]
Q_ALPHA = {3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}

for k in KEYS:
    name = SPECS[k].display_name
    fr = fold_results[k]

    # (a) full-grid omnibus test -- saved for the supplementary material
    res_all, ranks_all, nemenyi_all = friedman_nemenyi(fr, metric="f1", alpha=CONFIG.alpha)
    if "error" not in res_all:
        ranks_all.to_csv(RESULTS / f"stats_ranks_{k}.csv", index=False)
        if not nemenyi_all.empty:
            nemenyi_all.to_csv(RESULTS / f"stats_nemenyi_{k}.csv")

    # (b) shortlist -- this is what Figure 8 and Section 4.6 of the paper report
    s = fr[(fr.reduction == "LDA") & (fr.optimizer == "gwo")]
    top_classical = (s[~s.model.isin(QUANTUM)].groupby("model").f1.mean()
                       .nlargest(3).index.tolist())
    w = (s[s.model.isin(QUANTUM + top_classical)]
           .pivot_table(index="fold", columns="model", values="f1").dropna())
    chi2, p = stats.friedmanchisquare(*[w[c].values for c in w.columns])
    kk, n = w.shape[1], w.shape[0]
    cd = Q_ALPHA[kk] * np.sqrt(kk * (kk + 1) / (6.0 * n))
    ranks = (w.rank(axis=1, ascending=False).mean().sort_values()
               .rename("mean_rank").reset_index().rename(columns={"model": "config"}))
    ranks.to_csv(RESULTS / f"stats_ranks_shortlist_{k}.csv", index=False)
    within = int((ranks.mean_rank - ranks.mean_rank.iloc[0] <= cd).sum())

    print(f"\n=== {name} ===")
    print(f"  full grid : chi2={res_all.get('chi2', float('nan')):.1f}  "
          f"k={res_all.get('k_configs')}  CD={res_all.get('critical_difference', float('nan')):.2f}")
    print(f"  shortlist : chi2={chi2:.1f}  p={p:.1e}  k={kk}  n={n}  CD={cd:.2f}  "
          f"| within CD of best: {within}/{kk}")
    print(ranks.to_string(index=False))
    viz.plot_critical_difference(ranks, cd, figure_number=8, outdir=str(FIGURES),
        slug=f"critical_difference_{k}",
        title=f"Nemenyi critical difference — {name} (LDA, GWO)")

## 5. Comparison of search strategies

Two distinct comparisons are made, and they should not be conflated:

1. **Hyper-parameter search** — GWO vs PSO vs Random Search. Compared here, under an
   **identical objective-evaluation budget**. Without equal budgets a metaheuristic
   comparison is meaningless.
2. **Circuit-parameter training** — Adam vs SPSA vs COBYLA. That is Section 6, because it
   concerns the barren-plateau question, not hyper-parameter search.

Random Search is included as the standard baseline for hyper-parameter optimisation
(Bergstra & Bengio, 2012): a metaheuristic should be compared with uniform random sampling
at an equal budget.

In [ ]:
# One variational model and one kernel model answer the question "is GWO better
# than established alternatives?" while keeping the cost bounded -- this section
# re-runs the full nested CV once per optimizer.
# Add "QNN" back if the extra evidence is worth roughly three more hours.
OPT_MODELS = [m for m in ("VQC", "QSVC") if m in REGISTRY]

CKPT_OPT = RESULTS / "checkpoints_optimizers"; CKPT_OPT.mkdir(parents=True, exist_ok=True)

opt_prev_path = RESULTS / "folds_optimizer_comparison.csv"
opt_prev = load_folds(opt_prev_path) if opt_prev_path.exists() else None
if opt_prev is not None:
    print(f"restore source: {opt_prev_path.name} ({len(opt_prev)} rows)")

opt_frames = []
for k in KEYS:
    X, y = DATA[k]
    name = SPECS[k].display_name
    for opt in ("gwo", "pso", "random"):
        for m in OPT_MODELS:
            tag = f"{k}__{m}__{opt}"
            fpath = CKPT_OPT / f"{tag}.csv"
            if fpath.exists():
                opt_frames.append(load_folds(fpath))
                print(f"  [skip] {tag}", flush=True)
                continue
            if opt_prev is not None:
                prev = opt_prev[(opt_prev.dataset == name) & (opt_prev.model == m)
                                & (opt_prev.optimizer == opt)]
                if len(prev) == CONFIG.n_outer_folds:
                    prev.to_csv(fpath, index=False)
                    opt_frames.append(prev)
                    print(f"  [restored] {tag}", flush=True)
                    continue

            t0 = time.perf_counter()
            d = nested_cv(X, y, REGISTRY[m], CONFIG, reduction="lda",
                          n_components=CONFIG.lda_components, optimizer=opt,
                          n_jobs=CONFIG.n_jobs, verbose=False)
            d["dataset"] = name
            d.to_csv(fpath, index=False)
            opt_frames.append(d)
            print(f"  [ok]   {tag:<32s} F1={d.f1.mean():.3f} "
                  f"({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)

OPT_DF = pd.concat(opt_frames, ignore_index=True)
OPT_DF.to_csv(RESULTS / "folds_optimizer_comparison.csv", index=False)

opt_summary = summarise(OPT_DF, by=("dataset", "model", "optimizer"))
opt_summary.to_csv(RESULTS / "table_optimizer_comparison.csv", index=False)
print(opt_summary[["dataset", "model", "optimizer", "f1_mean", "f1_std", "n_folds"]]
      .to_string(index=False))

# Is GWO significantly better than PSO / Random Search at an equal budget?
for k in KEYS:
    sub = OPT_DF[OPT_DF.dataset == SPECS[k].display_name]
    pc = paired_comparison(sub, "f1", group_cols=("model", "optimizer"),
                           alpha=CONFIG.alpha)
    pc = pc[pc.config_A.str.contains("gwo") | pc.config_B.str.contains("gwo")]
    print(f"\n=== {SPECS[k].display_name}: GWO vs alternatives ===")
    print(pc[["config_A", "config_B", "mean_diff", "p_holm",
              "cliffs_delta", "significant"]].to_string(index=False))

## 6. Barren plateaus and vanishing gradients

### 6.1 Gradient-variance scaling

We measure `Var[∂⟨Z₀⟩/∂θ_k]` over Haar-random initialisations using **parameter-shift**
gradients, and fit `Var ~ exp(-α n)` (McClean et al., 2018). The index `k` is sampled
uniformly: parameter 0 of `StronglyEntanglingLayers` is an `RZ` on wire 0, which commutes
with the measured `Z₀`, so its gradient is *structurally* zero and would fabricate a
"vanishing gradient" unrelated to expressivity.

**Interpretation.** `α > 0` with a high R² indicates that the gradient variance decays
exponentially with qubit count, i.e. a barren plateau for this ansatz.

In [ ]:
# Output: barren_plateau_scan.csv / barren_plateau_fits.csv  (delete to recompute, ~1 h)
QUBIT_RANGE = (2, 3, 4, 5, 6, 7, 8)
DEPTH_RANGE = (1, 2, 4, 6)
N_GRAD_SAMPLES = 60 if CONFIG is SMOKE else 200

scan_path = RESULTS / "barren_plateau_scan.csv"
if scan_path.exists():
    scan = pd.read_csv(scan_path)
    print(f"[skip] loaded {scan_path.name} ({len(scan)} rows) -- delete it to recompute")
else:
    scan = gradient_variance_scan(qubit_range=QUBIT_RANGE, depth_range=DEPTH_RANGE,
                                  n_samples=N_GRAD_SAMPLES, feature_map="zz",
                                  diff_method="parameter-shift", verbose=True)
    scan.to_csv(scan_path, index=False)

fits = fit_decay(scan)
fits.to_csv(RESULTS / "barren_plateau_fits.csv", index=False)
print("\n", fits.to_string(index=False))

viz.plot_gradient_variance(scan, fits, figure_number=6, outdir=str(FIGURES))

### 6.2 Adam vs SPSA vs COBYLA on an identical circuit

Same architecture, same data, same initialisation seed — the only factor varying is the
optimiser that trains the circuit parameters.

In [ ]:
# Output: optimizer_trajectories.csv  (delete to recompute, ~5-15 min)
# A trajectories file without the `final_loss_full` column (full-batch loss
# after training, the common comparison metric) is recomputed.
from qmlgwo.config import get_rng as _get_rng

traj_path = RESULTS / "optimizer_trajectories.csv"
valid = traj_path.exists() and "final_loss_full" in pd.read_csv(traj_path, nrows=1).columns

if valid:
    traj = pd.read_csv(traj_path)
    print(f"[skip] loaded {traj_path.name} -- delete it to recompute")
else:
    if traj_path.exists():
        print(f"[recompute] {traj_path.name} is in the old, non-comparable format")
    k0 = KEYS[0]
    X0, y0 = DATA[k0]
    # Preprocess WITHOUT resampling so X and y stay aligned: this cell studies the
    # optimiser, not the class-imbalance strategy.
    prep, _ = build_pipeline(X0, "passthrough", reduction="pca",
                             n_components=CONFIG.pca_components,
                             balance="none", angle_encode=True, seed=CONFIG.master_seed)
    Xt = np.asarray(prep.fit_transform(X0, y0), dtype=float)
    yt = np.asarray(y0)
    # Balanced subsample keeps the comparison cheap and class-balanced.
    _rng = _get_rng("resampling", 0)
    n_per = min(200, int(np.bincount(yt).min()))
    idx = np.concatenate([_rng.choice(np.flatnonzero(yt == c), n_per, replace=False)
                          for c in (0, 1)])
    _rng.shuffle(idx)
    Xt, yt = Xt[idx], yt[idx]
    print(f"trajectory study on {SPECS[k0].display_name}: {len(yt)} samples, "
          f"{Xt.shape[1]} features -> {min(Xt.shape[1], CONFIG.max_qubits)} qubits")
    traj = optimizer_trajectories(Xt, yt, n_layers=3, n_epochs=CONFIG.n_epochs,
                                  learning_rate=CONFIG.learning_rate,
                                  feature_map="angle", seed=CONFIG.master_seed,
                                  max_qubits=CONFIG.max_qubits)
    traj.to_csv(traj_path, index=False)

traj_summary = summarise_trajectories(traj)
traj_summary.to_csv(RESULTS / "table_optimizer_trajectories.csv", index=False)
print(traj_summary.round(4).to_string(index=False))
viz.plot_optimizer_comparison(traj, figure_number=7, outdir=str(FIGURES))

### 6.3 What GWO can and cannot claim

GWO tunes **hyper-parameters** (depth, encoding repetitions, entanglement, learning rate).
It does not train circuit parameters — Adam/SPSA/COBYLA do. So GWO cannot "avoid barren
plateaus" in the sense of McClean et al.

The defensible claim, quantified below, is narrower: *the search prefers architectures
whose measured gradient variance is higher than the average over the search range.* If
`claim_supported` is `False`, that sentence must come out of the manuscript.

In [ ]:
evidence = []
for k in KEYS:
    for m in ("VQC", "QNN"):
        e = plateau_avoidance_evidence(
            fold_results[k][fold_results[k].optimizer == "gwo"], scan, model_name=m)
        if not e.empty:
            e["dataset"] = SPECS[k].display_name
            evidence.append(e)

if evidence:
    evidence = pd.concat(evidence, ignore_index=True)
    evidence.to_csv(RESULTS / "table_plateau_evidence.csv", index=False)
    print(evidence.to_string(index=False))
else:
    print("No GWO-selected depths recorded — run Section 3 first.")

## 7. Class-imbalance ablation

Random undersampling discards majority-class data, which on these imbalance ratios can be a
large fraction of the training set. Rather than defending the choice rhetorically, we
measure it: undersampling vs SMOTE vs no balancing, everything else held fixed. The
resampler sits **inside** the CV pipeline, so test folds are never rebalanced.

In [ ]:
# Output: folds_imbalance_ablation.csv  (delete to recompute, ~1-2 h)
ABLATION_MODELS = [m for m in ("VQC", "QSVC", "RandomForest") if m in REGISTRY]
abl_path = RESULTS / "folds_imbalance_ablation.csv"

if abl_path.exists():
    ABL = pd.read_csv(abl_path)
    print(f"[skip] loaded {abl_path.name} ({len(ABL)} rows) -- delete it to recompute")
else:
    abl_frames = []
    for strategy in ("undersample", "smote", "none"):
        cfg_s = ExperimentConfig(**{**CONFIG.to_dict(), "balance_strategy": strategy})
        for k in KEYS:
            X, y = DATA[k]
            for name in ABLATION_MODELS:
                d = nested_cv(X, y, build_registry(cfg_s)[name], cfg_s, reduction="lda",
                              n_components=cfg_s.lda_components, optimizer=None,
                              n_jobs=cfg_s.n_jobs, verbose=False)
                d["balance"] = strategy
                d["dataset"] = SPECS[k].display_name
                abl_frames.append(d)
    ABL = pd.concat(abl_frames, ignore_index=True)
    ABL.to_csv(abl_path, index=False)

abl_summary = summarise(ABL, metrics=("f1", "recall", "precision", "balanced_accuracy", "mcc"),
                        by=("dataset", "model", "balance"))
abl_summary.to_csv(RESULTS / "table_imbalance_ablation.csv", index=False)
print(abl_summary[["dataset", "model", "balance", "f1_mean", "f1_std",
                   "recall_mean", "precision_mean", "mcc_mean"]].to_string(index=False))

## 8. Figures

The files written below are per-dataset panels named `Figure_<n>_<slug>.{png,pdf}`. The
manuscript figures are built by `make_paper_figures.py` from the saved results (and, for
Figure 8, from the SHAP panels written here). File prefix and manuscript figure:

| File | Manuscript figure |
|---|---|
| `Figure_2_gwo_convergence_*` | Figure 4 (GWO convergence) |
| `Figure_3_confusion_matrices_*` | Figure 3 (confusion matrices) |
| `Figure_4_shap_summary_*` | Figure 8 (SHAP summaries) |
| `Figure_5_performance_*` | Figure 2 (F1 with 95% CI) |
| `Figure_6_barren_plateau` | Figure 5 (gradient variance) |
| `Figure_7_optimizer_comparison` | Figure 6 (circuit optimisers) |
| `Figure_8_critical_difference_*` | Figure 7 (critical-difference diagrams) |

Figure 1 (the evaluation protocol) is drawn by `make_paper_figures.py`.

In [ ]:
for k in KEYS:
    name = SPECS[k].display_name
    df = fold_results[k]
    gwo_lda = df[(df.optimizer == "gwo") & (df.reduction == "LDA")]

    viz.plot_convergence(gwo_lda, figure_number=2, outdir=str(FIGURES),
                         slug=f"gwo_convergence_{k}", title=f"GWO convergence — {name}")

    viz.plot_confusion_matrices(aggregate_confusion(gwo_lda, ("model",)),
                                figure_number=3, outdir=str(FIGURES),
                                slug=f"confusion_matrices_{k}")

    viz.plot_performance(summary_tables[k], metric="f1", figure_number=5,
                         outdir=str(FIGURES), slug=f"performance_{k}",
                         title=name, hue="reduction")

### 8.1 SHAP (manuscript Figure 8)

SHAP is computed over the **entire fitted pipeline**, so attributions are expressed in the
original feature space. Explaining only the classifier would, under LDA, attribute
everything to the single axis `LD1`.

**Model explained: VQC.** Ranked by F1 alone, SVC comes first on three datasets, but its
ROC-AUC on HR Promotion is 0.455 — its ranking is near-random there, and explaining a
near-random ranker is not informative. VQC is a quantum model with sound ROC-AUC on all
four datasets (0.668–0.755), and the paper is about quantum classifiers. The configuration
explained is the one GWO selected in the fold whose outer F1 is closest to the median.

Output: `Figure_4_shap_summary_<dataset>` and `shap_importance_<dataset>.csv`
(delete the PNG to recompute).

In [ ]:
import ast, shap
import matplotlib.pyplot as plt
from qmlgwo.config import get_rng

MODEL_FOR_SHAP = "VQC"
N_BACKGROUND, N_EXPLAIN, N_COALITIONS = 40, 80, 300

for k in KEYS:
    name = SPECS[k].display_name
    stem = FIGURES / f"Figure_4_shap_summary_{k}"
    if Path(f"{stem}.png").exists():
        print(f"[skip] {stem.name}.png exists -- delete it to recompute")
        continue

    X, y = DATA[k]
    fr = fold_results[k]
    sel = fr[(fr.model == MODEL_FOR_SHAP) & (fr.reduction == "LDA") & (fr.optimizer == "gwo")]
    # representative configuration: the fold whose outer F1 is closest to the median
    row = sel.iloc[(sel.f1 - sel.f1.median()).abs().argsort().iloc[0]]
    params = row.best_params if isinstance(row.best_params, dict) else ast.literal_eval(row.best_params)
    print(f"{name}: fold {int(row.fold)} (F1={row.f1:.3f}) -> {params}")

    pipe, _ = build_pipeline(X, REGISTRY[MODEL_FOR_SHAP].make(params, seed=CONFIG.master_seed),
                             reduction="lda", n_components=CONFIG.lda_components,
                             balance=CONFIG.balance_strategy, angle_encode=True,
                             seed=CONFIG.master_seed)
    rng = get_rng("shap", 0)
    fit_idx = rng.choice(len(X), min(len(X), CONFIG.max_train_samples), replace=False)
    pipe.fit(X.iloc[fit_idx], y[fit_idx])

    bg = X.iloc[rng.choice(len(X), N_BACKGROUND, replace=False)]
    ex = X.iloc[rng.choice(len(X), N_EXPLAIN, replace=False)]
    f = lambda a: pipe.predict_proba(pd.DataFrame(a, columns=X.columns))[:, 1]

    t0 = time.perf_counter()
    sv = shap.KernelExplainer(f, bg, silent=True).shap_values(ex, nsamples=N_COALITIONS, silent=True)

    # SHAP's colorbar is incompatible with the constrained-layout engine that
    # qmlgwo.viz enables globally -- this was why Figure 4 failed to save before.
    with plt.rc_context({"figure.constrained_layout.use": False}):
        plt.figure(figsize=(7.5, 5.0))
        shap.summary_plot(sv, ex, show=False, plot_size=None)
        plt.title(f"SHAP — {MODEL_FOR_SHAP}, {name}", fontsize=11)
        plt.savefig(f"{stem}.png", dpi=600, bbox_inches="tight")
        plt.savefig(f"{stem}.pdf", bbox_inches="tight")
        plt.close()

    imp = pd.Series(np.abs(sv).mean(0), index=X.columns).sort_values(ascending=False)
    imp.to_csv(RESULTS / f"shap_importance_{k}.csv", header=["mean_abs_shap"])
    print(f"   top features: {imp.head(5).round(4).to_dict()}")
    print(f"   saved {stem}.png / .pdf  ({time.perf_counter() - t0:.0f} s)")

## 9. Export and reproducibility manifest

Everything needed to re-derive the tables: fold-level scores, summaries,
statistical tests, the exact configuration, and the package versions used.

In [ ]:
manifest = {
    "notebook": NOTEBOOK,
    "datasets": [SPECS[k].display_name for k in KEYS],
    "config": CONFIG.to_dict(),
    "environment": environment_manifest(),
    "n_fold_records": int(len(ALL_FOLDS)),
    "models": MODELS,
    "reductions": [f"{r.upper()} (k={k})" for r, k in REDUCTIONS],
    "files": sorted(p.name for p in RESULTS.glob("*")),
    "figures": sorted(p.name for p in FIGURES.glob("*.png")),
}
(RESULTS / "MANIFEST.json").write_text(json.dumps(manifest, indent=2, default=str))

print(json.dumps({k: v for k, v in manifest.items()
                  if k not in ("config", "environment")}, indent=2, default=str))
print(f"\nResults -> {RESULTS.resolve()}")
print(f"Figures -> {FIGURES.resolve()}")

---

### Outputs and where they appear in the manuscript

| Output (in `results/notebook_<A|B>/`) | Manuscript |
|---|---|
| `folds_all.csv` (one row per outer fold) | source of every table and figure; Supplemental Data S1 (A) and S2 (B) |
| `table_datasets.csv` | Table 2 |
| `table_gwo_settings.csv`, `table_search_space.csv` | Tables 3 and 4 |
| `table_results_<dataset>.csv`, `summary_<dataset>.csv` | Tables 5–8, Figure 2 |
| `table_gwo_effect.csv` | Table 9, Section 4.2 |
| `table_optimizer_comparison.csv` | Table 10 |
| `barren_plateau_scan.csv`, `barren_plateau_fits.csv` | Table 11, Figure 5 |
| `optimizer_trajectories.csv` | Table 12, Figure 6 |
| `table_imbalance_ablation.csv` | Table 14 |
| `stats_pairwise_<dataset>.csv`, `stats_ranks_<dataset>.csv`, `stats_nemenyi_<dataset>.csv` | Sections 4.1–4.6, Figure 7 |
| `shap_importance_<dataset>.csv` | Section 4.8, Figure 8 |
| `config_and_environment.json`, `MANIFEST.json` | Section 3.13 |

Tables 13 and 15 are aggregates of `folds_all.csv`. The manuscript figures are assembled by
`make_paper_figures.py` from these files.
